# Temporal Deep Learning for Crop Yield Prediction

**B.Tech project | Real data | Google Colab ready**

Real crop statistics are downloaded from FAOSTAT and real environmental variables from NASA POWER. The notebook trains an LSTM on multi-year sequences. Optional MODIS NDVI integration is included as an extension because Earth Engine requires authentication.

In [ ]:
!pip -q install pandas numpy requests scikit-learn tensorflow matplotlib
import os, zipfile, glob, requests, warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
warnings.filterwarnings("ignore"); np.random.seed(42); tf.random.set_seed(42)


## 1. Download official FAOSTAT crop-yield data

In [ ]:
URL="https://bulks-faostat.fao.org/production/Production_Crops_Livestock_E_All_Data_(Normalized).zip"
ZIP="faostat_crops.zip"; OUT="faostat_data"
if not os.path.exists(ZIP):
    r=requests.get(URL,timeout=180); r.raise_for_status(); open(ZIP,"wb").write(r.content)
os.makedirs(OUT,exist_ok=True)
with zipfile.ZipFile(ZIP) as z: z.extractall(OUT)
files=glob.glob(OUT+"/**/*.csv",recursive=True)
print(files[:5])


In [ ]:
crop_file=next(f for f in files if "All_Data" in os.path.basename(f))
raw=pd.read_csv(crop_file,encoding="latin-1")
india=raw[raw["Area"].eq("India")].copy()
major=["Wheat","Rice","Maize"]
d=india[india["Item"].isin(major) & india["Element"].astype(str).str.lower().str.contains("yield")].copy()
d=d.rename(columns={"Year":"year","Item":"crop","Value":"yield_raw","Unit":"yield_unit"})[["year","crop","yield_raw","yield_unit"]]
d["year"]=pd.to_numeric(d.year,errors="coerce"); d["yield_raw"]=pd.to_numeric(d.yield_raw,errors="coerce"); d=d.dropna()
d["yield_tonnes_per_ha"]=np.where(d.yield_unit.astype(str).str.contains("100 mg/ha|hg/ha",case=False,regex=True),d.yield_raw/10000,d.yield_raw)
d=d.sort_values(["crop","year"]); print(d.groupby("crop").year.agg(["min","max","count"]))


## 2. Download real environmental data from NASA POWER

In [ ]:
LAT,LON=22.9734,78.6569
start,end=int(d.year.min()),int(d.year.max())
params="T2M,PRECTOTCORR,RH2M,ALLSKY_SFC_SW_DWN"
u=f"https://power.larc.nasa.gov/api/temporal/daily/point?parameters={params}&community=AG&longitude={LON}&latitude={LAT}&start={start}0101&end={end}1231&format=JSON"
r=requests.get(u,timeout=120); r.raise_for_status()
w=pd.DataFrame(r.json()["properties"]["parameter"]); w.index=pd.to_datetime(w.index,format="%Y%m%d",errors="coerce"); w=w.replace(-999,np.nan)
a=w.resample("YE").agg({"T2M":"mean","PRECTOTCORR":"sum","RH2M":"mean","ALLSKY_SFC_SW_DWN":"mean"}).reset_index(); a["year"]=a["index"].dt.year; a=a.drop(columns="index")


## 3. Merge real data and create temporal sequences

In [ ]:
df=d[["year","crop","yield_tonnes_per_ha"]].merge(a,on="year",how="inner")
df=pd.get_dummies(df,columns=["crop"],dtype=int).sort_values("year").reset_index(drop=True)
df.to_csv("real_crop_environment_dataset.csv",index=False)
features=["T2M","PRECTOTCORR","RH2M","ALLSKY_SFC_SW_DWN"]+[x for x in df if x.startswith("crop_")]
WINDOW=5; X=[]; y=[]; yrs=[]
for cc in [x for x in df if x.startswith("crop_")]:
    p=df[df[cc]==1].sort_values("year")
    for i in range(WINDOW,len(p)):
        X.append(p[features].iloc[i-WINDOW:i].values); y.append(p.yield_tonnes_per_ha.iloc[i]); yrs.append(p.year.iloc[i])
X=np.asarray(X,float); y=np.asarray(y,float); yrs=np.asarray(yrs)
cut=np.quantile(yrs,.8); tr=yrs<=cut; sc=StandardScaler(); n,steps,nf=X[tr].shape
Xtr=sc.fit_transform(X[tr].reshape(-1,nf)).reshape(n,steps,nf); Xte=sc.transform(X[~tr].reshape(-1,nf)).reshape(X[~tr].shape)
ytr,yte=y[tr],y[~tr]; print(Xtr.shape,Xte.shape,"split",cut)


## 4. Train LSTM and evaluate

In [ ]:
model=Sequential([LSTM(64,return_sequences=True,input_shape=(WINDOW,len(features))),Dropout(.2),LSTM(32),Dropout(.2),Dense(16,activation="relu"),Dense(1)])
model.compile(optimizer="adam",loss="mse",metrics=["mae"])
h=model.fit(Xtr,ytr,validation_split=.2,epochs=100,batch_size=16,callbacks=[EarlyStopping(patience=12,restore_best_weights=True)],verbose=1)
pred=model.predict(Xte,verbose=0).ravel()
print("MAE",mean_absolute_error(yte,pred)); print("RMSE",mean_squared_error(yte,pred)**.5); print("R2",r2_score(yte,pred))
plt.figure(figsize=(8,4)); plt.plot(yte,marker="o",label="Actual"); plt.plot(pred,marker="x",label="Predicted"); plt.title("Actual vs Predicted Yield"); plt.xlabel("Test observation"); plt.ylabel("Tonnes/hectare"); plt.grid(); plt.legend(); plt.show()
model.save("crop_yield_lstm.keras")


## Optional: Real MODIS satellite NDVI
To strengthen the project, add `MODIS/061/MOD13Q1` NDVI through Google Earth Engine. This requires a one-time Earth Engine authorization and Google Cloud project. Do not replace it with synthetic NDVI data.

In [ ]:
# Optional extension
# !pip -q install earthengine-api geemap
# import ee
# ee.Authenticate()
# ee.Initialize(project="YOUR_GOOGLE_CLOUD_PROJECT_ID")
# modis = ee.ImageCollection("MODIS/061/MOD13Q1")
# Extract annual mean NDVI for a selected agricultural region and merge by year.
